# Read and write files · Student

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 20. Read and write files

Move data between pandas and common local file formats.

Before you start: Know DataFrame construction, column selection, and paths.

Official reference: https://pandas.pydata.org/docs/user_guide/io.html

### M20C01. Read and export CSV

CSV stores text rows separated by delimiters. read_csv creates a DataFrame and attempts to infer column types. to_csv writes the selected data; index=False avoids saving the pandas row index as an extra column.

**Read:** What does DATA_DIR / "students.csv" build?

Your explanation: _write here_.

**Predict:** How many students score at least 85?

Your explanation: _write here_.

**Explain:** Why keep Student_ID as text?

Your explanation: _write here_.



In [ ]:
# Predict before running: How many students score at least 85?
df = pd.read_csv(DATA_DIR / "students.csv", dtype={"Student_ID": "string"})
result = df.loc[df["Score"].ge(85), ["Student_ID", "Name", "Score"]]
result.to_csv(OUTPUT_DIR / "high_scores.csv", index=False)
print(result)
print("Saved outputs/high_scores.csv")

**Your turn:** Read only Student_ID, Name, and Attendance. Save students with Attendance below 85 to attendance_followup.csv.

For the check, keep these result names: `result`.

**Hint:** Use usecols when reading and .lt(85) when filtering.

**Watch for this:** CSV does not preserve every pandas dtype. Check types and missing-value rules when reloading.

In [ ]:
# Change this code to complete the task.
# Question: Read only Student_ID, Name, and Attendance. Save students with Attendance below 85 to attendance_followup.csv.
df = pd.read_csv(DATA_DIR / "students.csv", dtype={"Student_ID": "string"})
result = df.loc[df["Score"].ge(85), ["Student_ID", "Name", "Score"]]
result.to_csv(OUTPUT_DIR / "high_scores.csv", index=False)
print(result)
print("Saved outputs/high_scores.csv")

**Quick check:** Why use index=False on export?

1. To sort the records
2. To omit the pandas row index
3. To delete Student_ID

**Explain what you learned:** Inspect imports and choose export columns deliberately.

### M20C02. Move records through JSON

JSON can store a list of records, each represented by a dictionary. orient="records" makes this structure explicit. Re-reading a file checks that the exported shape matches what you intended.

**Read:** What does one JSON record represent?

Your explanation: _write here_.

**Predict:** How many dictionaries are written?

Your explanation: _write here_.

**Explain:** Why inspect the exported JSON as text?

Your explanation: _write here_.



In [ ]:
# Predict before running: How many dictionaries are written?
df = pd.read_json(DATA_DIR / "students.json", orient="records")
result = df[["Student_ID", "Campus", "Score"]].head(3)
result.to_json(OUTPUT_DIR / "student_preview.json", orient="records", indent=2)
print(result)
print((OUTPUT_DIR / "student_preview.json").read_text())

**Your turn:** Export Student_ID and Program for the first two students, then read the saved JSON back into result.

For the check, keep these result names: `result`.

**Hint:** Write with orient="records" and read with the same orientation.

**Watch for this:** File formats can change date and nullable dtype representations; a round trip needs validation.

In [ ]:
# Change this code to complete the task.
# Question: Export Student_ID and Program for the first two students, then read the saved JSON back into result.
df = pd.read_json(DATA_DIR / "students.json", orient="records")
result = df[["Student_ID", "Campus", "Score"]].head(3)
result.to_json(OUTPUT_DIR / "student_preview.json", orient="records", indent=2)
print(result)
print((OUTPUT_DIR / "student_preview.json").read_text())

**Quick check:** What does orient="records" produce?

1. A list of row dictionaries
2. One image
3. An Excel worksheet

**Explain what you learned:** Choose a file orientation that matches the receiving system.

### M20C03. Select an Excel worksheet

An Excel workbook can contain several sheets. sheet_name selects one by name. The openpyxl engine handles the provided .xlsx workbook and is listed in the installation requirements.

**Read:** What is the difference between the file and the sheet?

Your explanation: _write here_.

**Predict:** How many rows are in the summary?

Your explanation: _write here_.

**Explain:** Why use a new output filename?

Your explanation: _write here_.



In [ ]:
# Predict before running: How many rows are in the summary?
df = pd.read_excel(DATA_DIR / "students.xlsx", sheet_name="Students", engine="openpyxl")
result = df.groupby("Campus", as_index=False)["Score"].mean()
result.to_excel(OUTPUT_DIR / "campus_report.xlsx", sheet_name="Summary", index=False, engine="openpyxl")
print(result.round(2))

**Your turn:** Read the Students sheet. Store a summary in result with columns Program and Students, where Students counts rows per program. Export it to program_counts.xlsx, sheet Counts.

For the check, keep these result names: `result`.

**Hint:** groupby("Program").size() counts rows; name the count column.

**Watch for this:** If openpyxl is missing, install the provided requirements before running this cell.

In [ ]:
# Change this code to complete the task.
# Question: Read the Students sheet. Store a summary in result with columns Program and Students, where Students counts rows per program. Export it to program_counts.xlsx, sheet Counts.
df = pd.read_excel(DATA_DIR / "students.xlsx", sheet_name="Students", engine="openpyxl")
result = df.groupby("Campus", as_index=False)["Score"].mean()
result.to_excel(OUTPUT_DIR / "campus_report.xlsx", sheet_name="Summary", index=False, engine="openpyxl")
print(result.round(2))

**Quick check:** Which argument chooses a worksheet?

1. sheet_name
2. index=False
3. dtype alone

**Explain what you learned:** Excel I/O needs a workbook path and the correct worksheet.

### M20C04. Query a small SQLite table

SQL retrieves data from tables using a query. This example creates a temporary in-memory database from the fictional data. The parameter placeholder passes a value separately from the query text.

**Read:** What does :memory: mean?

Your explanation: _write here_.

**Predict:** Which student appears first?

Your explanation: _write here_.

**Explain:** Why use a placeholder and params?

Your explanation: _write here_.



In [ ]:
# Predict before running: Which student appears first?
with sqlite3.connect(":memory:") as connection:
    students().to_sql("students", connection, index=False, if_exists="replace")
    result = pd.read_sql_query(
        "SELECT Student_ID, Name, Score FROM students WHERE Score >= ? ORDER BY Score DESC",
        connection, params=(85,))
print(result)

**Your turn:** Use SQL to calculate mean Score and student count for each Campus. Store the returned DataFrame in result with columns Campus, Mean_Score, and Students, ordered by Campus.

For the check, keep these result names: `result`.

**Hint:** Use GROUP BY Campus with AVG(Score) AS Mean_Score and COUNT(*) AS Students.

**Watch for this:** Only use if_exists="replace" here on the temporary classroom table; on a real database it replaces the existing table.

In [ ]:
# Change this code to complete the task.
# Question: Use SQL to calculate mean Score and student count for each Campus. Store the returned DataFrame in result with columns Campus, Mean_Score, and Students, ordered by Campus.
with sqlite3.connect(":memory:") as connection:
    students().to_sql("students", connection, index=False, if_exists="replace")
    result = pd.read_sql_query(
        "SELECT Student_ID, Name, Score FROM students WHERE Score >= ? ORDER BY Score DESC",
        connection, params=(85,))
print(result)

**Quick check:** What does read_sql_query return?

1. A DataFrame
2. A database server
3. An Excel file

**Explain what you learned:** Pandas can receive SQL results without loading every source row.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates